# 3. Baseline Models

## Purpose
Establish a performance benchmark by training and cross validating multiple
models on the preprocessed data. All decisions about which model to take
forward are made using cross validation scores only.

## Inputs
- `data/processed/` — preprocessed train and test features and labels
- `data/raw/` — raw data reloaded for CatBoost
- `config.yaml` — baseline model settings
- `artifacts/preprocessor.pkl` — fitted preprocessing pipeline

## Outputs
- `models/baseline/` — all trained baseline models
- `artifacts/baseline_trainer.pkl` — trainer state including leaderboard
  and CV results for all models

## Decisions for the user
- Set `baseline.run_all` in `config.yaml` — true runs all models for the
  detected task, false runs only the models listed under `baseline.models`
- Set `baseline.cv_folds` in `config.yaml` — typically 5 for most projects
- Review the leaderboard and select which models to take forward to tuning
  via `tuning.models_to_tune` in `config.yaml`

## Important note
The preprocessor passed to the trainer is unfitted. It is fitted inside
each CV fold on training data only, preventing data leakage into the
validation fold.

In [1]:
import pandas as pd
import numpy as np
import joblib
import copy

from src.config.settings import load_config
from src.config.paths import (
    PROCESSED_DATA_DIR,
    BASELINE_MODELS_DIR,
    ARTIFACTS_DIR
)
from src.features.pipeline import FeatureEngineeringPipeline
from src.models.training.trainer import ModelTrainer
from src.models.training.model_registry import MODEL_REGISTRY
from src.models.training.save_load import save_model
from src.data.load_data import load_raw_data, prepare_data

## 1. Load config

In [5]:
config = load_config()

TARGET = config["target"]
BASELINE_CONFIG = config["baseline"]
PIPELINE_CONFIG = config["pipeline"]
SPLIT_CONFIG = config["split"]

print(f"Target:         {TARGET}")
print(f"Run all models: {BASELINE_CONFIG['run_all']}")
print(f"CV folds:       {BASELINE_CONFIG['cv_folds']}")

Target:         good
Run all models: True
CV folds:       5


## 2. Load processed data

In [6]:
X_train = pd.read_csv(PROCESSED_DATA_DIR / "X_train.csv")
X_test = pd.read_csv(PROCESSED_DATA_DIR / "X_test.csv")
y_train = pd.read_csv(PROCESSED_DATA_DIR / "y_train.csv").squeeze()
y_test = pd.read_csv(PROCESSED_DATA_DIR / "y_test.csv").squeeze()

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape:  {X_test.shape}")

X_train shape: (1087, 11)
X_test shape:  (272, 11)


## 3. Load raw data for CatBoost
The same split parameters from config are used to ensure
identical train/test splits across all models.

In [8]:
from src.data.load_data import prepare_data
from sklearn.model_selection import train_test_split

df = load_raw_data(na_values=config.get("na_values", []))
X, y = prepare_data(df, config)

stratify_col = y if SPLIT_CONFIG["stratify"] else None

X_train_raw, X_test_raw, _, _ = train_test_split(
    X, y,
    test_size=SPLIT_CONFIG["test_size"],
    random_state=SPLIT_CONFIG["random_state"],
    stratify=stratify_col
)

print(f"X_train_raw shape: {X_train_raw.shape}")
print(f"X_test_raw shape:  {X_test_raw.shape}")

240 instances have been dropped
X_train_raw shape: (1087, 11)
X_test_raw shape:  (272, 11)


## 4. Initialise preprocessor
The preprocessor must be unfitted at this stage. The trainer will deep copy
and fit it inside each CV fold to prevent data leakage.

In [9]:
preprocessor = FeatureEngineeringPipeline(config=PIPELINE_CONFIG)

## 5. Initialise trainer

In [10]:
trainer = ModelTrainer(
    model_registry=MODEL_REGISTRY,
    random_state=BASELINE_CONFIG["random_state"]
)

## 6. Train baseline models
Trains all models appropriate for the detected task type using stratified
k-fold cross validation. Metrics are averaged across folds.

In [11]:
trainer.fit(
    X=X_train_raw,
    y=y_train.values,
    preprocessor=preprocessor,
    cv=BASELINE_CONFIG["cv_folds"],
    run_all=BASELINE_CONFIG["run_all"],
    model_subset=BASELINE_CONFIG.get("models", None)
)

Task detected: classification

Training: logistic_regression

Training: random_forest_classifier

Training: extra_trees_classifier

Training: xgboost_classifier

Training: lightgbm_classifier


/home/kigr/miniforge3/envs/wine-quality/lib/python3.11/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
/home/kigr/miniforge3/envs/wine-quality/lib/python3.11/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
/home/kigr/miniforge3/envs/wine-quality/lib/python3.11/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
/home/kigr/miniforge3/envs/wine-quality/lib/python3.11/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X


Training: catboost_classifier

Training complete.


## 7. Leaderboard
Review CV scores for all models. Use this to decide which models to take
forward to hyperparameter tuning — update `tuning.models_to_tune` in
`config.yaml` accordingly.

In [12]:
trainer.leaderboard_

,model_name,primary_metric_name,primary_metric_value,roc_auc,f1,precision,recall,accuracy,log_loss,rmse,mae,r2,training_time_sec
0,catboost_classifier,roc_auc,0.838716,0.838716,0.764603,0.764664,0.764589,0.765425,0.492237,None,None,None,1.633013
1,xgboost_classifier,roc_auc,0.837974,0.837974,0.750907,0.751131,0.751223,0.751617,0.493084,None,None,None,130.682133
2,extra_trees_classifier,roc_auc,0.836040,0.836040,0.757692,0.759295,0.757451,0.758978,0.503024,None,None,None,4.025779
3,random_forest_classifier,roc_auc,0.829937,0.829937,0.758095,0.758750,0.758227,0.759007,0.508417,None,None,None,4.627476
4,lightgbm_classifier,roc_auc,0.826860,0.826860,0.753337,0.754139,0.753195,0.754369,0.507560,None,None,None,243.141958
5,logistic_regression,roc_auc,0.816108,0.816108,0.744726,0.744845,0.745348,0.745178,0.526484,None,None,None,0.226579


## 8. Identify best model
The best model by CV score is identified here for reference. This is the
benchmark to beat in the feature engineering and tuning notebooks.

In [13]:
best_model_name = trainer.leaderboard_.iloc[0]["model_name"]
best_model = trainer.models_[best_model_name]

print(f"Best model from CV: {best_model_name}")
print(f"CV {trainer.leaderboard_.iloc[0]['primary_metric_name']}: "
      f"{trainer.leaderboard_.iloc[0]['primary_metric_value']:.4f}")

Best model from CV: catboost_classifier
CV roc_auc: 0.8387


## 9. Save baseline models and trainer state
All models are saved individually to `models/baseline/`. The trainer state
is saved to `artifacts/` so downstream notebooks can access the leaderboard
and CV results for comparison.

In [14]:
BASELINE_MODELS_DIR.mkdir(parents=True, exist_ok=True)

for model_name, model in trainer.models_.items():
    save_model(model, BASELINE_MODELS_DIR / f"{model_name}.pkl")

print(f"All baseline models saved to {BASELINE_MODELS_DIR}")

# %%
# 10. Save Trainer State
joblib.dump(trainer, ARTIFACTS_DIR / "baseline_trainer.pkl")
print(f"Trainer saved to {ARTIFACTS_DIR / 'baseline_trainer.pkl'}")

Saving model: /home/kigr/wine-quality/models/baseline/logistic_regression.pkl

Saving model: /home/kigr/wine-quality/models/baseline/random_forest_classifier.pkl

Saving model: /home/kigr/wine-quality/models/baseline/extra_trees_classifier.pkl

Saving model: /home/kigr/wine-quality/models/baseline/xgboost_classifier.pkl

Saving model: /home/kigr/wine-quality/models/baseline/lightgbm_classifier.pkl

Saving model: /home/kigr/wine-quality/models/baseline/catboost_classifier.pkl

All baseline models saved to /home/kigr/wine-quality/models/baseline
Trainer saved to /home/kigr/wine-quality/artifacts/baseline_trainer.pkl
